# 2-Stage 딥페이크 음성 탐지 시스템
## Multi-Feature Extraction + Temporal Sequence Modeling

### 연구 배경 및 동기

기존 1-Stage 모델(AttentionAudioClassifier)은 각 오디오 파일에서 MFCC 13차원의
**시간축 평균(mean pooling)** 벡터 하나만을 추출하여 분류에 사용했다.
이 방식은 시간적 변화 패턴(temporal dynamics)을 완전히 소실시킨다.

딥페이크 음성과 실제 음성의 핵심 차이는 **시간축 상의 미세한 불일치**에 있다:
- TTS 음성: 프로소디(운율)의 부자연스러운 전이, 포먼트 궤적의 비연속성
- Voice Conversion: 변환 과정에서 발생하는 프레임 간 불연속적 아티팩트
- 실제 음성: 자연스러운 코아티큘레이션(co-articulation)과 연속적 스펙트럴 변이

### 2-Stage 아키텍처 설계 철학

**Stage 1 (Feature Extractor):**
기존 학습된 AttentionAudioClassifier의 분류 헤드를 제거하고,
SE-Attention이 적용된 중간 표현(256차원 벡터)을 프레임 단위로 추출한다.
추가로, MFCC 외에 LFCC, Mel-Spectrogram, Spectral Contrast, Delta/Delta-Delta 등
다양한 음향 특징을 병렬로 추출하여 멀티-뷰 표현을 구성한다.

**Stage 2 (Temporal Classifier):**
프레임 단위 임베딩 시퀀스를 Transformer Encoder 또는 BiLSTM에 입력하여
시간적 패턴을 학습한다. 참고 논문:
- Xie et al. (2024) EAT: ResNet + Transformer Encoder + BiLSTM 결합
- LSTM-AE-DRDE (2025): Attention-enhanced LSTM with contrastive learning
- DK-CAST (2025): Multi-level supervision including embeddings and phoneme posteriors
- DLSA (2025): MFCC + CQT 2-stage MHA fusion, ASVspoof 2019 LA EER 0.68%

### 참고 문헌
1. Todisco et al., "ASVspoof 2019: Future horizons in spoofed and fake audio detection," 2019
2. Xie et al., "EAT: ResNet + Transformer + BiLSTM framework," 2024
3. Zaman et al., "Hybrid transformer architectures with diverse audio features," IEEE Access, 2024
4. Zhang et al., "Audio deepfake detection: what has been achieved and what lies ahead," Sensors, 2025
5. LSTM-AE-DRDE, "LSTM autoencoder with dynamic residual encoding," Scientific Reports, 2025
6. Shaaban, "Audio Deepfake Detection Using Deep Learning (Siamese CNN)," Engineering Reports, 2025
7. DLSA, "Spoof detection with dynamic learnable sparse attention and tri-modal fusion," PLOS ONE, 2025
8. DK-CAST, "Dynamic knowledge condensation with audio-selective transformer," Discover Computing, 2025
9. Deepfake voice detection with E2E transformer and acoustic feature fusion, Electronics, 2025


---
## 셀 1: 라이브러리 임포트 및 전역 설정

In [1]:
import os
import sys
import math
import copy
import warnings
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from glob import glob
from tqdm import tqdm
from collections import Counter, OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import (
    Dataset, DataLoader, WeightedRandomSampler, TensorDataset
)
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score
)
from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings('ignore')

# ==============================================================
# 전역 설정
# ==============================================================
# [수정] 데이터셋 경로 - 실제 폴더명 'DATASET'으로 변경
# 폴더 구조: DATASET/{train,val,test}/{real,fake}/*.wav
BASE_DIR = 'DATASET'

# [수정] 클래스 수 상수 - 데이터셋이 real/fake 2-class로 변경됨
# 기존: 3-class (real/fake/tts) 였으나 데이터셋에 tts 폴더 없음
# 노트북 전체에서 이 상수를 참조하도록 통일
NUM_CLASSES = 2

# 오디오 전처리 파라미터
SAMPLE_RATE = 16000       # ASVspoof 표준 샘플링 레이트
SEGMENT_LENGTH = 4.0
OVERLAP = 0.25

# 특징 추출 파라미터
N_MFCC = 13               # MFCC 계수 (ASVspoof 표준)
N_LFCC = 20               # LFCC 계수 (고주파 해상도가 더 높으므로 MFCC보다 많이 사용)
N_MELS = 40               # Mel-Spectrogram 필터뱅크 수
N_FFT = 512               # FFT 윈도우 (32ms at 16kHz)
HOP_LENGTH = 160           # 홉 길이 (10ms at 16kHz, ASVspoof 표준)
WIN_LENGTH = 400           # 윈도우 길이 (25ms at 16kHz)

# 시퀀스 모델링 파라미터
MAX_SEQ_LEN = 400         # 최대 프레임 수 (4초 / 10ms = 400)
FRAME_EMBEDDING_DIM = 256 # Stage 1 출력 임베딩 차원

# 학습 파라미터
BATCH_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 20

# 시드 고정
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# [수정] device 선언을 cell 2에 통합
# 기존: cell 6에서 처음 device 정의 - cell 8 실행 시 device 미정의 위험
# 수정: 전역 설정 셀에서 한 번만 정의하고 이후 셀에서 공통 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    # [수정] TF32 활성화 (Ampere 이후 GPU에서 FP32 대비 속도 향상)
    # RTX 4070 Ti(Ada Lovelace) 등에서 행렬/conv 연산 가속
    # 정확도 손실 거의 없음 (mantissa 10bit)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"PyTorch   : {torch.__version__}")
print(f"device    : {device}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"CUDA ver  : {torch.version.cuda}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"TF32      : {torch.backends.cuda.matmul.allow_tf32}")


PyTorch   : 2.5.1
device    : cuda
GPU       : NVIDIA GeForce RTX 4070 Ti
CUDA ver  : 12.1
VRAM      : 11.99 GB
TF32      : True


---
## 셀 2: 다중 음향 특징 추출 함수 정의

교수님 피드백: "MFCC 특징 외에 다른 특징들도 함께 고려해도 좋을 것 같아요"

각 특징이 포착하는 음향적 성질:

### MFCC (Mel-Frequency Cepstral Coefficients)
- **물리적 의미**: 인간 청각의 비선형 주파수 감도(mel scale)를 반영한 스펙트럼 포락선
- **딥페이크 탐지 관련성**: 보코더가 생성하는 멜 필터뱅크 출력의 과도한 평활화를 감지
- **한계**: 고주파 대역(4kHz 이상)의 해상도가 낮아 보코더 아티팩트 포착이 제한적

### LFCC (Linear-Frequency Cepstral Coefficients)
- **물리적 의미**: 선형 주파수 스케일의 필터뱅크 기반 캡스트럼 계수
- **딥페이크 탐지 관련성**: MFCC와 달리 고주파 대역을 균등하게 커버하여
  neural vocoder의 고주파 아티팩트(금속성 잡음, 앨리어싱)를 더 잘 포착
- **근거**: ASVspoof 2019/2021에서 LFCC가 MFCC보다 일관적으로 우수한 성능 기록

### Mel-Spectrogram
- **물리적 의미**: 시간-주파수 에너지 분포를 멜 스케일로 압축한 2D 표현
- **딥페이크 탐지 관련성**: 프레임 간 에너지 전이의 연속성/불연속성을 보존

### Spectral Contrast
- **물리적 의미**: 각 서브밴드에서 스펙트럼 피크와 밸리의 에너지 차이
- **딥페이크 탐지 관련성**: 실제 음성은 하모닉 구조로 높은 콘트라스트,
  보코더 합성 음성은 스펙트럼이 더 평탄하여 콘트라스트가 낮은 경향

### Delta 및 Delta-Delta (시간 미분 계수)
- **물리적 의미**: 특징의 1차/2차 시간 미분, 즉 변화 속도와 가속도
- **딥페이크 탐지 관련성**: TTS 음성의 부자연스럽게 매끄러운 전이 vs 실제 음성의 자연스러운 변동
- **근거**: ASVspoof 기본 시스템에서 MFCC + Delta + Delta-Delta 조합이 표준


In [2]:
def compute_lfcc(y, sr, n_lfcc=N_LFCC, n_fft=N_FFT,
                 hop_length=HOP_LENGTH, win_length=WIN_LENGTH):
    # LFCC (Linear Frequency Cepstral Coefficients) 추출
    #
    # MFCC가 멜 스케일(로그 주파수)을 사용하는 반면,
    # LFCC는 선형 주파수 스케일의 필터뱅크를 사용한다.
    #
    # 이론적 배경:
    #   선형 필터뱅크는 모든 주파수 대역을 균등한 대역폭으로 커버한다.
    #   따라서 고주파 대역(4kHz~8kHz)의 해상도가 MFCC보다 높다.
    #   Neural vocoder(WaveNet, WaveRNN, HiFi-GAN 등)는 고주파 대역에서
    #   앨리어싱, 금속성 노이즈, 부자연스러운 감쇠 등의 아티팩트를 남긴다.
    #   LFCC는 이러한 고주파 아티팩트를 더 정밀하게 포착할 수 있다.
    #
    # 구현:
    #   1. STFT -> 파워 스펙트로그램
    #   2. 선형 주파수 간격 삼각 필터뱅크 적용 (멜 스케일 대신)
    #   3. 로그 에너지 계산
    #   4. DCT로 캡스트럴 계수 추출
    #
    # 반환: (n_lfcc, T) 형태의 LFCC 행렬

    from scipy.fft import dct

    # 파워 스펙트로그램
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length,
                            win_length=win_length)) ** 2

    # 선형 주파수 간격 필터뱅크 생성
    n_filters = n_lfcc * 2
    freq_bins = n_fft // 2 + 1
    fmin = 0
    fmax = sr / 2

    # 선형 간격 중심 주파수 (멜 필터뱅크와의 핵심 차이)
    center_freqs = np.linspace(fmin, fmax, n_filters + 2)
    bin_indices = np.floor((n_fft + 1) * center_freqs / sr).astype(int)

    # 삼각 필터뱅크 구성
    filterbank = np.zeros((n_filters, freq_bins))
    for i in range(n_filters):
        left = bin_indices[i]
        center = bin_indices[i + 1]
        right = bin_indices[i + 2]
        if center > left:
            filterbank[i, left:center] = (
                np.arange(left, center) - left
            ) / (center - left)
        if right > center:
            filterbank[i, center:right] = (
                right - np.arange(center, right)
            ) / (right - center)

    # 필터뱅크 적용 -> 로그 -> DCT
    filter_energies = np.dot(filterbank, S)
    log_energies = np.log(filter_energies + 1e-10)
    lfcc = dct(log_energies, type=2, axis=0, norm='ortho')[:n_lfcc]

    return lfcc


def extract_multi_features_per_frame(audio_path, sr=SAMPLE_RATE,
                                      max_frames=MAX_SEQ_LEN):
    # 하나의 오디오 파일에서 프레임 단위 다중 특징 벡터를 추출한다.
    #
    # 추출되는 특징 (각 프레임 t에 대해):
    #   - MFCC (13D): 멜 스케일 스펙트럼 포락선
    #   - MFCC Delta (13D): 1차 시간 미분 (변화 속도)
    #     -> TTS는 파라미터 보간으로 delta가 비정상적으로 매끄러움
    #   - MFCC Delta-Delta (13D): 2차 시간 미분 (변화 가속도)
    #     -> 실제 음성은 코아티큘레이션으로 복잡한 가속 패턴
    #   - LFCC (20D): 선형 주파수 캡스트럼 (고주파 해상도 강화)
    #   - Mel-Spectrogram (40D): 멜 필터뱅크 에너지
    #   - Spectral Contrast (7D): 서브밴드 피크-밸리 에너지 차이
    #     -> 실제 음성은 하모닉 구조로 높은 콘트라스트,
    #        보코더 음성은 스펙트럼이 더 평탄하여 낮은 콘트라스트
    #   합계: 106차원 프레임 벡터
    #
    # 반환: (T, 106) numpy array 또는 None

    try:
        # [수정] duration=SEGMENT_LENGTH+0.5 상한 추가
        # 기존: librosa.load(audio_path, sr=sr, mono=True)
        # 문제: 파일 전체를 RAM에 올림
        #       shape (257, 1575) 같은 긴 파일이 STFT 중간 배열로
        #       수십 MiB를 할당하면서 MemoryError 발생
        # 수정: SEGMENT_LENGTH(4.0)+0.5초로 잘라 단일 파일 RAM 점유 상한 고정
        #       MAX_SEQ_LEN=400프레임(4초) 기준이므로 초과분은 어차피 잘림
        y, _ = librosa.load(
            audio_path, sr=sr, mono=True,
            duration=SEGMENT_LENGTH + 0.5
        )

        # 0.5초 미만 제외
        if len(y) < sr * 0.5:
            return None

        # MFCC: 멜 스케일 스펙트럼 포락선 (ASVspoof 표준 13차원)
        mfcc = librosa.feature.mfcc(
            y=y, sr=sr, n_mfcc=N_MFCC,
            n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH
        )  # (13, T)

        # MFCC Delta: 인접 프레임 변화율 (width=5: 전후 2프레임)
        mfcc_delta = librosa.feature.delta(mfcc, width=5)  # (13, T)

        # MFCC Delta-Delta: 변화율의 변화율 (가속도)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2, width=5)  # (13, T)

        # LFCC: 고주파 보코더 아티팩트 탐지용 선형 주파수 캡스트럼
        lfcc = compute_lfcc(y, sr)  # (20, T)

        # Mel-Spectrogram: 시간-주파수 에너지 분포
        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=sr, n_mels=N_MELS,
            n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH
        )
        log_mel = librosa.power_to_db(mel_spec, ref=np.max)  # (40, T)

        # Spectral Contrast: 서브밴드 피크-밸리 에너지 차이
        # 하모닉 구조가 있는 실제 음성 = 높은 콘트라스트
        # 보코더 음성 = 스펙트럼이 더 평탄 = 낮은 콘트라스트
        spectral_contrast = librosa.feature.spectral_contrast(
            y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_bands=6  # 6 서브밴드 + 1 밸리 = 7차원
        )  # (7, T)

        # 프레임 수 통일 (각 특징의 T가 약간 다를 수 있음)
        min_frames_count = min(
            mfcc.shape[1], lfcc.shape[1],
            log_mel.shape[1], spectral_contrast.shape[1]
        )

        mfcc = mfcc[:, :min_frames_count]
        mfcc_delta = mfcc_delta[:, :min_frames_count]
        mfcc_delta2 = mfcc_delta2[:, :min_frames_count]
        lfcc = lfcc[:, :min_frames_count]
        log_mel = log_mel[:, :min_frames_count]
        spectral_contrast = spectral_contrast[:, :min_frames_count]

        # 모든 특징 결합: (106, T)
        combined = np.concatenate([
            mfcc,               # 13: 멜 스케일 스펙트럼 포락선
            mfcc_delta,         # 13: 시간적 변화 속도
            mfcc_delta2,        # 13: 시간적 변화 가속도
            lfcc,               # 20: 선형 주파수 캡스트럼
            log_mel,            # 40: 멜 스펙트로그램 에너지
            spectral_contrast,  #  7: 서브밴드 피크-밸리 차이
        ], axis=0)

        # 전치: (T, 106) - 시퀀스 모델 입력 형식
        combined = combined.T

        # 시퀀스 길이 제한
        if combined.shape[0] > max_frames:
            combined = combined[:max_frames]

        return combined

    except Exception as e:
        print(f"특징 추출 오류 ({audio_path}): {e}")
        return None


# 특징 추출 함수 테스트
print("다중 특징 추출 함수 결과")
test_signal = np.random.randn(SAMPLE_RATE * 3)
sf.write(r'C:\Users\ipl\Desktop\test\DATASET\train\real\R_000000.wav', test_signal, SAMPLE_RATE)
test_features = extract_multi_features_per_frame(r'C:\Users\ipl\Desktop\test\DATASET\train\real\R_000000.wav')
if test_features is not None:
    print(f"  출력 형태: {test_features.shape}")
    print(f"  프레임 수 (T): {test_features.shape[0]}")
    print(f"  특징 차원 (D): {test_features.shape[1]}")
    feat_breakdown = "MFCC(13) + Delta(13) + Delta2(13) + LFCC(20) + Mel(40) + Contrast(7)"
    print(f"  특징 구성: {feat_breakdown} = {13+13+13+20+40+7}")


다중 특징 추출 함수 결과
  출력 형태: (301, 106)
  프레임 수 (T): 301
  특징 차원 (D): 106
  특징 구성: MFCC(13) + Delta(13) + Delta2(13) + LFCC(20) + Mel(40) + Contrast(7) = 106


---
## 셀 3: Stage 1 모델 정의 - 기존 AttentionAudioClassifier 기반 임베딩 추출기

교수님 피드백: "현재 모델을 통해 각 오디오 시퀀스를 벡터 형태로 만들고"

기존 학습된 AttentionAudioClassifier의 가중치를 재활용한다.
핵심: 분류 헤드(classifier)를 제거하고, SE-Attention이 적용된
중간 표현(256차원)을 각 프레임의 "음성 품질 임베딩"으로 사용한다.

이론적 근거:
1. **전이 학습**: 기존 모델이 학습한 판별적 표현을 시퀀스 모델 입력으로 활용
2. **SE 블록**: 이미 "딥페이크 탐지에 중요한 특징 채널"에 대한 어텐션을 학습
3. **2-Stage 분리**: 특징 추출과 시간적 모델링을 독립 최적화, 해석 가능성 향상

참고: DK-CAST (2025) 교사 모델 중간 표현 전이, Wav2Vec2 fine-tuning 접근법


In [3]:
# ================================================================
# Stage 1-A: 기존 모델 구조 정의 (가중치 로드용)
# ================================================================

class SEBlock(nn.Module):
    # Squeeze-and-Excitation 블록 (채널 어텐션, Hu et al., 2018)
    #
    # 각 채널의 글로벌 정보를 squeeze하고, 채널 간 상호의존성을 excitation으로 모델링.
    # 수식: z = GlobalAvgPool(x), s = sigmoid(W2 * ReLU(W1 * z)), output = x * s
    #
    # 딥페이크 탐지에서의 역할:
    #   MFCC 13개 계수 중 딥페이크 구분에 변별력 있는 계수(고차 캡스트럼 등)에
    #   자동으로 더 높은 가중치를 할당한다.

    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        y = F.relu(self.fc1(x))
        y = torch.sigmoid(self.fc2(y))
        return x * y


class AttentionAudioClassifier(nn.Module):
    # 기존 1-Stage ASV 기반 딥페이크 분류기 (원본 구조 보존)
    # Stage 1에서 가중치 로드 및 중간 표현 추출 용도

    def __init__(self, input_dim, num_classes=3):
        super().__init__()
        self.fc1 = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512),
            nn.ReLU(), nn.Dropout(0.4)
        )
        self.se1 = SEBlock(512)
        self.fc2 = nn.Sequential(
            nn.Linear(512, 384), nn.BatchNorm1d(384),
            nn.ReLU(), nn.Dropout(0.35)
        )
        self.se2 = SEBlock(384)
        self.fc3 = nn.Sequential(
            nn.Linear(384, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.se3 = SEBlock(256)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.se1(self.fc1(x))
        x = self.se2(self.fc2(x))
        x = self.se3(self.fc3(x))
        return self.classifier(x)


# ================================================================
# Stage 1-B: 임베딩 추출기 (분류 헤드 제거)
# ================================================================

class Stage1EmbeddingExtractor(nn.Module):
    # 기존 AttentionAudioClassifier에서 classifier 헤드를 제거하고,
    # SE-Attention이 적용된 256차원 중간 표현을 출력한다.
    #
    # 동작: MFCC (13D) -> FC(512)+SE -> FC(384)+SE -> FC(256)+SE -> 출력 (256D)
    #
    # 이 256차원 벡터의 의미:
    #   기존 모델이 Real/Fake/TTS 분류를 위해 학습한 판별적 공간의 표현.
    #   SE 블록이 적용되어 "딥페이크 탐지에 중요한 특징"이 강조된 상태.
    #   각 프레임에 대한 "음성 품질 임베딩"으로 기능한다.
    #
    # 이론적 근거:
    #   전이 학습(Transfer Learning) - 기존 학습된 판별적 표현을 시퀀스 모델 입력으로 활용
    #   DK-CAST (2025): 교사 모델 중간 표현을 학생 모델에 전달하는 것과 유사
    #   Wav2Vec2 fine-tuning: 사전학습 모델 상위 레이어만 미세조정하는 것과 유사

    def __init__(self, input_dim=13):
        super().__init__()
        self.fc1 = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512),
            nn.ReLU(), nn.Dropout(0.4)
        )
        self.se1 = SEBlock(512)
        self.fc2 = nn.Sequential(
            nn.Linear(512, 384), nn.BatchNorm1d(384),
            nn.ReLU(), nn.Dropout(0.35)
        )
        self.se2 = SEBlock(384)
        self.fc3 = nn.Sequential(
            nn.Linear(384, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.se3 = SEBlock(256)

    def forward(self, x):
        # x: (B, 13) 또는 (B, T, 13) 형태
        # 3D 입력: (B, T, 13) -> 프레임별 독립 처리 -> (B, T, 256)
        if x.dim() == 3:
            B, T, D = x.shape
            x = x.reshape(B * T, D)
            x = self.se1(self.fc1(x))
            x = self.se2(self.fc2(x))
            x = self.se3(self.fc3(x))
            return x.reshape(B, T, -1)
        else:
            x = self.se1(self.fc1(x))
            x = self.se2(self.fc2(x))
            x = self.se3(self.fc3(x))
            return x

    @classmethod
    def from_pretrained(cls, checkpoint_path, input_dim=13, device='cpu'):
        # 기존 AttentionAudioClassifier 가중치에서 분류 헤드를 제외하고 전이
        extractor = cls(input_dim=input_dim).to(device)
        state_dict = torch.load(checkpoint_path, map_location=device)

        # classifier.* 가중치 제외
        filtered = OrderedDict()
        for key, val in state_dict.items():
            if not key.startswith('classifier'):
                filtered[key] = val

        extractor.load_state_dict(filtered, strict=False)
        extractor.eval()
        print(f"Stage 1 임베딩 추출기 로드 완료 ({checkpoint_path})")
        print(f"  전이 파라미터: {len(filtered)}개, 제외: {len(state_dict)-len(filtered)}개 (classifier)")
        return extractor


# 기존 모델 가중치 로드 시도
# [수정] device는 cell 2에서 이미 정의됨 (중복 제거)

PRETRAINED_PATH = 'best_asvspoof_model.pt'
if os.path.exists(PRETRAINED_PATH):
    stage1_extractor = Stage1EmbeddingExtractor.from_pretrained(
        PRETRAINED_PATH, input_dim=N_MFCC, device=device
    )
    USE_PRETRAINED_STAGE1 = True
else:
    print(f"경고: '{PRETRAINED_PATH}' 파일 없음. Stage 1 랜덤 초기화.")
    print("(기존 모델 학습 후 이 셀을 다시 실행하면 전이 학습 적용)")
    stage1_extractor = Stage1EmbeddingExtractor(input_dim=N_MFCC).to(device)
    USE_PRETRAINED_STAGE1 = False


경고: 'best_asvspoof_model.pt' 파일 없음. Stage 1 랜덤 초기화.
(기존 모델 학습 후 이 셀을 다시 실행하면 전이 학습 적용)


---
## 셀 4: 프레임 단위 특징 추출 파이프라인

두 종류의 프레임 벡터를 추출하여 결합한다:

**A. 다중 음향 특징 (106차원)** - 물리적 신호 성질 직접 반영

**B. Stage 1 임베딩 (256차원)** - 데이터 기반 판별적 패턴 (선택적)

**결합 전략: Late Concatenation (106 + 256 = 362차원)**

이론적 근거 (다중 뷰 학습):
- 음향 특징과 학습된 임베딩은 상보적(complementary) 정보를 제공
- DLSA (2025)에서 MFCC + CQT + raw waveform 3-modal fusion으로 EER 0.68% 달성
- 기존 모델이 없으면 106차원 음향 특징만 사용


In [4]:
import gc
import pickle
import hashlib

# ==============================================================
# 체크포인트 유틸리티
# ==============================================================

def get_checkpoint_path(base_dir, split, cls_name):
    '''
    체크포인트 디렉터리 경로를 결정한다.
    base_dir + split + cls_name 조합으로 고유 식별자를 만들어
    서로 다른 데이터셋/split이 같은 디렉터리를 공유하지 않도록 한다.
    '''
    key = f"{base_dir}_{split}_{cls_name}"
    uid = hashlib.md5(key.encode()).hexdigest()[:8]
    return f"checkpoint_seq_{split}_{cls_name}_{uid}_chunks"


def save_chunk(chunk_dir, chunk_idx, sequences, labels, lengths):
    '''
    현재 버퍼(sequences)를 청크 pkl 파일로 저장하고 RAM에서 비울 수 있도록 한다.

    [수정 이유]
    기존: sequences 전체를 하나의 pkl에 담아 저장
    문제: sequences가 이미 RAM을 꽉 채운 상태에서 pickle.dump가
          직렬화 버퍼를 추가로 요구해 MemoryError 발생
    수정: checkpoint_every개 단위로 청크 파일을 분리 저장하고
          호출측에서 버퍼를 비워 RAM을 확보한다.
    '''
    os.makedirs(chunk_dir, exist_ok=True)
    chunk_path = os.path.join(chunk_dir, f"chunk_{chunk_idx:05d}.pkl")
    tmp_path = chunk_path + ".tmp"
    with open(tmp_path, 'wb') as f:
        pickle.dump(
            {'sequences': sequences, 'labels': labels, 'lengths': lengths},
            f, protocol=4
        )
    os.replace(tmp_path, chunk_path)


def save_meta(chunk_dir, processed_indices, skip_count, total_chunks):
    '''
    진행 상태(처리 완료 인덱스, 건너뜀 수, 청크 수)를 메타파일로 저장한다.
    sequences 데이터와 분리하여 재시작 시 메타만 빠르게 읽을 수 있다.
    '''
    meta_path = os.path.join(chunk_dir, "meta.pkl")
    tmp_path = meta_path + ".tmp"
    with open(tmp_path, 'wb') as f:
        pickle.dump(
            {
                'processed_indices': processed_indices,
                'skip_count': skip_count,
                'total_chunks': total_chunks,
            },
            f, protocol=4
        )
    os.replace(tmp_path, meta_path)


def load_checkpoint(chunk_dir):
    '''
    체크포인트를 로드한다.

    메타파일에서 진행 상태를 읽고,
    청크 파일들을 순서대로 읽어 sequences를 복구한다.
    파일이 없거나 손상된 경우 None을 반환해 처음부터 시작하도록 한다.

    [수정 이유 - 이전 버전 버그]
    이전: load_checkpoint가 반환한 sequences를 recovered_sequences에 담고,
          flush 후 비워진 buf_sequences와 합산 -> 마지막 청크 미만 분만 카운트
    수정: load_checkpoint에서 모든 청크를 합산해 완전한 sequences를 반환하고,
          호출측은 이를 그대로 all_sequences에 extend한다.
          buf는 새로 처리할 파일만 담는 용도로만 사용한다.
    '''
    meta_path = os.path.join(chunk_dir, "meta.pkl")
    if not os.path.exists(meta_path):
        return None
    try:
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)

        sequences, labels, lengths = [], [], []
        for i in range(meta['total_chunks']):
            chunk_path = os.path.join(chunk_dir, f"chunk_{i:05d}.pkl")
            with open(chunk_path, 'rb') as f:
                chunk = pickle.load(f)
            sequences.extend(chunk['sequences'])
            labels.extend(chunk['labels'])
            lengths.extend(chunk['lengths'])
            del chunk
            gc.collect()

        print(f"    청크 {meta['total_chunks']}개에서 {len(sequences)}개 복구")
        return {
            'processed_indices': meta['processed_indices'],
            'skip_count': meta['skip_count'],
            'total_chunks': meta['total_chunks'],
            'sequences': sequences,
            'labels': labels,
            'lengths': lengths,
        }
    except Exception as e:
        print(f"    체크포인트 로드 실패 ({chunk_dir}): {e}")
        print(f"    처음부터 다시 시작합니다.")
        return None


# ==============================================================
# Dataset / collate (변경 없음)
# ==============================================================

class DeepfakeSequenceDataset(Dataset):
    # 딥페이크 음성 탐지용 시퀀스 데이터셋
    # 각 오디오 -> 프레임 단위 특징 시퀀스로 저장
    # 가변 길이 시퀀스 지원 (collate_fn에서 패딩)

    def __init__(self, sequences, labels, lengths):
        self.sequences = sequences
        self.labels = labels
        self.lengths = lengths

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.FloatTensor(self.sequences[idx]),
            torch.LongTensor([self.labels[idx]]).squeeze(),
            self.lengths[idx]
        )


def collate_sequences(batch):
    # 가변 길이 시퀀스를 배치 내 최대 길이로 zero-padding.
    # 실제 길이 정보를 반환하여 Transformer/LSTM 마스킹에 활용.
    # pack_padded_sequence 요구사항: 길이 기준 내림차순 정렬.

    sequences, labels, lengths = zip(*batch)

    sorted_indices = sorted(range(len(lengths)),
                           key=lambda i: lengths[i], reverse=True)
    sequences = [sequences[i] for i in sorted_indices]
    labels = [labels[i] for i in sorted_indices]
    lengths = [lengths[i] for i in sorted_indices]

    padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)
    labels = torch.stack(labels)
    lengths = torch.LongTensor(lengths)

    return padded, labels, lengths


# ==============================================================
# 특징 추출 (청크 분리 체크포인트 + 메모리 안전)
# ==============================================================

def extract_dataset_sequences(base_dir, split, classes,
                               stage1_model=None, scaler_path=None,
                               device='cpu',
                               checkpoint_every=1000,
                               resume=True):
    '''
    처리 흐름:
      1. 오디오 로드
      2. 다중 음향 특징 추출 (106D per frame)
      3. (선택) Stage 1 모델로 MFCC 프레임별 임베딩 추출 (256D per frame)
      4. 두 특징 결합 -> (106+256 = 362D) 또는 (106D only)

    체크포인트 전략:
      checkpoint_every개마다 buf를 청크 파일로 flush하고 RAM을 비운다.
      항상 최대 checkpoint_every개분의 sequences만 RAM에 존재한다.
      재시작 시 청크 파일들을 순서대로 읽어 복구하고 미처리 파일만 이어서 처리한다.

    파라미터:
      checkpoint_every : 이 파일 수마다 청크 저장 및 RAM flush (기본 1000)
      resume           : True면 기존 체크포인트에서 이어서 시작
    '''

    mfcc_scaler = None
    if scaler_path and os.path.exists(scaler_path):
        mfcc_scaler = joblib.load(scaler_path)
        print(f"  MFCC 스케일러 로드: {scaler_path}")

    all_sequences = []
    all_labels = []
    all_lengths = []
    total_skip = 0

    for cls_name, label in classes.items():
        folder = os.path.join(base_dir, split, cls_name)
        if not os.path.exists(folder):
            print(f"    경고: {folder} 없음, 건너뜀...")
            continue

        files = sorted(glob(os.path.join(folder, '*.wav')) +
                       glob(os.path.join(folder, '*.mp3')))
        print(f"    {cls_name} ({label}): {len(files)}개 파일")

        chunk_dir = get_checkpoint_path(base_dir, split, cls_name)

        # 새로 처리할 파일의 버퍼 (checkpoint_every개까지만 쌓음)
        buf_sequences = []
        buf_labels = []
        buf_lengths = []
        skip_count = 0
        processed_indices = set()
        chunk_idx = 0

        # ----------------------------------------------------------
        # 체크포인트 복구
        #
        # 복구된 sequences는 all_sequences에 바로 extend한다.
        # buf는 새로 처리할 파일만 담는다.
        #
        # [이전 버전 버그]
        # recovered_sequences + buf_sequences 합산 시
        # buf가 flush 후 비워진 상태라 마지막 청크 미만 분만 카운트됨.
        # -> 26,492개 처리했는데 300개만 유효로 집계된 원인
        #
        # [수정]
        # 복구 즉시 all_sequences에 extend하고,
        # buf는 이번 실행에서 새로 추가되는 파일만 누적한다.
        # ----------------------------------------------------------
        if resume:
            ckpt = load_checkpoint(chunk_dir)
            if ckpt is not None:
                all_sequences.extend(ckpt['sequences'])
                all_labels.extend(ckpt['labels'])
                all_lengths.extend(ckpt['lengths'])
                skip_count = ckpt['skip_count']
                processed_indices = ckpt['processed_indices']
                chunk_idx = ckpt['total_chunks']
                print(f"    체크포인트 복구: {len(processed_indices)}개 완료, "
                      f"{len(ckpt['sequences'])}개 유효, {skip_count}개 건너뜀")

        for idx, fpath in enumerate(tqdm(files, desc=f"  {split}/{cls_name}", leave=False)):

            if idx in processed_indices:
                continue

            try:
                multi_feat = extract_multi_features_per_frame(fpath)
            except MemoryError:
                skip_count += 1
                processed_indices.add(idx)
                gc.collect()
                continue
            except Exception:
                skip_count += 1
                processed_indices.add(idx)
                continue

            if multi_feat is None:
                skip_count += 1
                processed_indices.add(idx)
                continue

            T = multi_feat.shape[0]

            y_loaded = None
            if stage1_model is not None:
                try:
                    y_loaded, _ = librosa.load(
                        fpath, sr=SAMPLE_RATE,
                        duration=SEGMENT_LENGTH + 0.5
                    )
                except MemoryError:
                    skip_count += 1
                    processed_indices.add(idx)
                    gc.collect()
                    continue
                except Exception:
                    y_loaded = None

            if stage1_model is not None and y_loaded is not None:
                try:
                    mfcc_raw = librosa.feature.mfcc(
                        y=y_loaded, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
                        n_fft=N_FFT, hop_length=HOP_LENGTH,
                        win_length=WIN_LENGTH
                    ).T
                    mfcc_raw = mfcc_raw[:T]

                    if mfcc_scaler is not None:
                        mfcc_raw = mfcc_scaler.transform(mfcc_raw)

                    with torch.no_grad():
                        mfcc_tensor = torch.FloatTensor(mfcc_raw).to(device)
                        if mfcc_tensor.shape[0] > 0:
                            embedding = stage1_model(
                                mfcc_tensor.unsqueeze(0)
                            ).squeeze(0).cpu().numpy()

                            min_t = min(multi_feat.shape[0], embedding.shape[0])
                            multi_feat = multi_feat[:min_t]
                            embedding = embedding[:min_t]

                            combined = np.concatenate(
                                [multi_feat, embedding], axis=1
                            )
                        else:
                            combined = multi_feat

                    del mfcc_tensor, mfcc_raw

                except MemoryError:
                    combined = multi_feat
                    gc.collect()
                except Exception:
                    combined = multi_feat
            else:
                combined = multi_feat

            if y_loaded is not None:
                del y_loaded

            buf_sequences.append(combined)
            buf_labels.append(label)
            buf_lengths.append(combined.shape[0])
            processed_indices.add(idx)

            # ----------------------------------------------------------
            # checkpoint_every개마다 청크 flush
            #
            # buf를 디스크에 저장하고 즉시 비워 RAM을 확보한다.
            # all_sequences에도 바로 extend하여 집계에서 누락되지 않도록 한다.
            # ----------------------------------------------------------
            if len(buf_sequences) >= checkpoint_every:
                save_chunk(chunk_dir, chunk_idx, buf_sequences, buf_labels, buf_lengths)
                save_meta(chunk_dir, processed_indices, skip_count, chunk_idx + 1)
                chunk_idx += 1
                # buf를 all_sequences에 합산한 뒤 비움
                all_sequences.extend(buf_sequences)
                all_labels.extend(buf_labels)
                all_lengths.extend(buf_lengths)
                buf_sequences = []
                buf_labels = []
                buf_lengths = []
                gc.collect()

        # 루프 종료 후 남은 버퍼 처리
        if buf_sequences:
            save_chunk(chunk_dir, chunk_idx, buf_sequences, buf_labels, buf_lengths)
            chunk_idx += 1
            all_sequences.extend(buf_sequences)
            all_labels.extend(buf_labels)
            all_lengths.extend(buf_lengths)
            buf_sequences = []
            buf_labels = []
            buf_lengths = []

        save_meta(chunk_dir, processed_indices, skip_count, chunk_idx)
        gc.collect()

        total_skip += skip_count
        print(f"    {cls_name} 완료: 누적 {len(all_sequences)}개 유효, "
              f"{skip_count}개 건너뜀, 체크포인트: {chunk_dir}/")

    feat_dim = all_sequences[0].shape[1] if all_sequences else 0
    print(f"  {split} 전체 완료: {len(all_sequences)}개, 차원: {feat_dim}")
    if total_skip > 0:
        print(f"  총 건너뛴 파일: {total_skip}개 (손상/메모리 오류)")

    return all_sequences, all_labels, all_lengths


# ==============================================================
# 클래스 매핑 및 실행
# ==============================================================

classes = {'real': 0, 'fake': 1}
class_names = ['real', 'fake']

print("=" * 60)
print("프레임 단위 다중 특징 추출 시작")
print("=" * 60)

# resume=True: 기존 체크포인트가 있으면 이어서 시작
# resume=False: 처음부터 새로 시작 (체크포인트 무시)
data_sequences = {}
for split in ['train', 'val', 'test']:
    print(f"\n{split} split 처리 중...")
    seqs, labs, lens = extract_dataset_sequences(
        BASE_DIR, split, classes,
        stage1_model=stage1_extractor if USE_PRETRAINED_STAGE1 else None,
        scaler_path='scaler_asvspoof.pkl' if USE_PRETRAINED_STAGE1 else None,
        device=device,
        checkpoint_every=1000,
        resume=True
    )
    data_sequences[split] = {
        'sequences': seqs, 'labels': labs, 'lengths': lens
    }
    gc.collect()

# 데이터 요약
print("\n" + "=" * 60)
print("시퀀스 데이터 요약")
print("=" * 60)
for split in ['train', 'val', 'test']:
    d = data_sequences[split]
    if d['sequences']:
        lens_arr = np.array(d['lengths'])
        print(f"  {split}: {len(d['sequences'])}개, "
              f"dim={d['sequences'][0].shape[1]}, "
              f"len min/mean/max={lens_arr.min()}/{lens_arr.mean():.0f}/{lens_arr.max()}")


프레임 단위 다중 특징 추출 시작

train split 처리 중...
    real (0): 38301개 파일
    청크 39개에서 38300개 복구
    체크포인트 복구: 38301개 완료, 38300개 유효, 1개 건너뜀


    real 완료: 누적 38300개 유효, 1개 건너뜀, 체크포인트: checkpoint_seq_train_real_9b06a812_chunks/


    fake (1): 88812개 파일
    청크 89개에서 88811개 복구
    체크포인트 복구: 88812개 완료, 88811개 유효, 1개 건너뜀


    fake 완료: 누적 127111개 유효, 1개 건너뜀, 체크포인트: checkpoint_seq_train_fake_8a850e9b_chunks/
  train 전체 완료: 127111개, 차원: 106
  총 건너뛴 파일: 2개 (손상/메모리 오류)



val split 처리 중...
    real (0): 4787개 파일
    청크 5개에서 4787개 복구
    체크포인트 복구: 4787개 완료, 4787개 유효, 0개 건너뜀


...

    fake 완료: 누적 15888개 유효, 0개 건너뜀, 체크포인트: checkpoint_seq_val_fake_9077ecb0_chunks/
  val 전체 완료: 15888개, 차원: 106

test split 처리 중...
    real (0): 4789개 파일


    청크 5개에서 4789개 복구
    체크포인트 복구: 4789개 완료, 4789개 유효, 0개 건너뜀


    real 완료: 누적 4789개 유효, 0개 건너뜀, 체크포인트: checkpoint_seq_test_real_f26b714d_chunks/
    fake (1): 11102개 파일


    청크 12개에서 11102개 복구
    체크포인트 복구: 11102개 완료, 11102개 유효, 0개 건너뜀


    fake 완료: 누적 15891개 유효, 0개 건너뜀, 체크포인트: checkpoint_seq_test_fake_f0a7faef_chunks/
  test 전체 완료: 15891개, 차원: 106

시퀀스 데이터 요약
  train: 127111개, dim=106, len min/mean/max=52/318/400
  val: 15888개, dim=106, len min/mean/max=55/318/400
  test: 15891개, dim=106, len min/mean/max=51/317/400


---
## 셀 5: 시퀀스 데이터 정규화 및 DataLoader 구성

### 정규화 전략
각 특징 차원에 대해 **전체 학습 데이터의 모든 프레임**을 기준으로
평균과 분산을 계산하고, 모든 split에 동일하게 적용한다.

### 시퀀스 SMOTE의 한계와 대안
- 시퀀스 데이터는 길이가 가변이므로 SMOTE 직접 적용이 어려움
- 대신 **클래스 가중치** + **WeightedRandomSampler**로 불균형 해결
- 학습 시 **SpecAugment**(시간/특징 마스킹)으로 추가 증강


In [5]:
# ==============================================================
# 시퀀스 데이터 정규화 (디스크 청크 기반 Welford 알고리즘)
#
# [수정 이유]
# RAM 15GB 환경에서 sequences가 31.88GB를 점유하므로
# np.concatenate + partial_fit 모두 추가 버퍼 할당에서 OOM 발생.
# sklearn partial_fit 내부도 float64 업캐스트로 추가 배열을 만들어 OOM.
#
# 수정 방향:
#   체크포인트 청크 파일에서 직접 읽어 fit -> transform -> 저장.
#   Welford 온라인 알고리즘으로 mean/var를 직접 누적하여
#   추가 배열 할당 없이 정규화 파라미터를 계산한다.
#
# 전제: cell 8의 체크포인트 청크 폴더가 존재해야 한다.
#       재시작 후 cell 8을 건너뛰고 이 셀을 바로 실행 가능.
# ==============================================================

import pickle

# cell 8 실행 로그에서 확인한 체크포인트 폴더명
# 폴더명이 다르면 실제 생성된 이름으로 수정할 것
CHUNK_DIRS = {
    'train': [
        'checkpoint_seq_train_real_9b06a812_chunks',
        'checkpoint_seq_train_fake_8a850e9b_chunks',
    ],
    'val': [
        'checkpoint_seq_val_real_f26d8108_chunks',
        'checkpoint_seq_val_fake_9077ecb0_chunks',
    ],
    'test': [
        'checkpoint_seq_test_real_f26b714d_chunks',
        'checkpoint_seq_test_fake_f0a7faef_chunks',
    ],
}

def iter_chunks(dirs):
    # 청크 디렉터리 목록에서 청크를 순서대로 yield
    # 한 번에 청크 하나만 메모리에 올림
    for chunk_dir in dirs:
        meta_path = os.path.join(chunk_dir, 'meta.pkl')
        if not os.path.exists(meta_path):
            print(f"  경고: {chunk_dir} 없음, 건너뜀")
            continue
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        for ci in range(meta['total_chunks']):
            chunk_path = os.path.join(chunk_dir, f'chunk_{ci:05d}.pkl')
            with open(chunk_path, 'rb') as f:
                chunk = pickle.load(f)
            yield chunk
            del chunk

# --------------------------------------------------------------
# Step 1: Welford 알고리즘으로 mean/std 계산
#
# partial_fit 대신 직접 구현하는 이유:
#   sklearn 내부에서 (X - T) 배열을 float64로 추가 할당해 OOM 발생.
#   Welford는 프레임 단위 스칼라 갱신이므로 추가 배열 없음.
#   수치 안정성을 위해 누적 변수만 float64 사용.
# --------------------------------------------------------------
print("[1/3] Welford 알고리즘으로 mean/std 계산...")

n_total = np.int64(0)
mean = None
M2 = None

for chunk in iter_chunks(CHUNK_DIRS['train']):
    frames = np.concatenate(
        [s.astype(np.float32) for s in chunk['sequences']], axis=0
    )  # (N, 106) float32
    for frame in frames:
        f64 = frame.astype(np.float64)
        n_total += 1
        if mean is None:
            mean = f64.copy()
            M2 = np.zeros_like(f64)
        else:
            delta = f64 - mean
            mean += delta / n_total
            M2 += delta * (f64 - mean)
    del frames
    gc.collect()

variance = M2 / n_total
std = np.sqrt(variance) + 1e-8

print(f"  완료: {n_total:,} 프레임, dim={len(mean)}")

# StandardScaler 호환 객체 구성 (transform 메서드 사용 가능)
seq_scaler = StandardScaler()
seq_scaler.mean_ = mean.astype(np.float64)
seq_scaler.scale_ = std.astype(np.float64)
seq_scaler.var_ = variance.astype(np.float64)
seq_scaler.n_features_in_ = int(len(mean))
seq_scaler.n_samples_seen_ = int(n_total)

joblib.dump(seq_scaler, 'seq_scaler_2stage.pkl')
print("  스케일러 저장: seq_scaler_2stage.pkl")
del M2, variance
gc.collect()

# --------------------------------------------------------------
# Step 2: 청크 파일을 정규화하여 덮어씀
# float32 연산으로 RAM 절약
# --------------------------------------------------------------
print("\n[2/3] 청크 파일 정규화 (in-place)...")

mean_f32 = mean.astype(np.float32)
std_f32 = std.astype(np.float32)

for split, dirs in CHUNK_DIRS.items():
    for chunk_dir in dirs:
        meta_path = os.path.join(chunk_dir, 'meta.pkl')
        if not os.path.exists(meta_path):
            continue
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        for ci in range(meta['total_chunks']):
            chunk_path = os.path.join(chunk_dir, f'chunk_{ci:05d}.pkl')
            with open(chunk_path, 'rb') as f:
                chunk = pickle.load(f)
            chunk['sequences'] = [
                (s.astype(np.float32) - mean_f32) / std_f32
                for s in chunk['sequences']
            ]
            tmp = chunk_path + '.tmp'
            with open(tmp, 'wb') as f:
                pickle.dump(chunk, f, protocol=4)
            os.replace(tmp, chunk_path)
            del chunk
            gc.collect()
        print(f"  {chunk_dir} 완료")

del mean_f32, std_f32, mean, std
gc.collect()

# --------------------------------------------------------------
# Step 3: data_sequences 재구성 (정규화된 청크에서 로드)
# 기존 sequences를 완전히 비우고 float32로만 올림
# --------------------------------------------------------------
print("\n[3/3] data_sequences 재구성...")

data_sequences = {
    split: {'sequences': [], 'labels': [], 'lengths': []}
    for split in ['train', 'val', 'test']
}

for split, dirs in CHUNK_DIRS.items():
    for chunk in iter_chunks(dirs):
        data_sequences[split]['sequences'].extend(chunk['sequences'])
        data_sequences[split]['labels'].extend(chunk['labels'])
        data_sequences[split]['lengths'].extend(chunk['lengths'])
        gc.collect()
    n = len(data_sequences[split]['sequences'])
    mb = sum(s.nbytes for s in data_sequences[split]['sequences']) / 1024**2
    print(f"  {split}: {n:,}개, {mb:.0f} MB")

print("\n정규화 완료")
gc.collect()

# ==============================================================
# Dataset 및 DataLoader 생성
# ==============================================================

train_dataset = DeepfakeSequenceDataset(
    data_sequences['train']['sequences'],
    data_sequences['train']['labels'],
    data_sequences['train']['lengths']
)
val_dataset = DeepfakeSequenceDataset(
    data_sequences['val']['sequences'],
    data_sequences['val']['labels'],
    data_sequences['val']['lengths']
)
test_dataset = DeepfakeSequenceDataset(
    data_sequences['test']['sequences'],
    data_sequences['test']['labels'],
    data_sequences['test']['lengths']
)

# [수정] 클래스 가중치 - NUM_CLASSES에 맞춰 동적 계산
# 기존: 3-class 하드코딩 (real/fake/tts), tts 3배 가중
# 수정: 2-class 데이터셋(real/fake)에 맞춰 균형 가중치 사용
#       fake 클래스가 일반적으로 데이터가 많으므로 자연스럽게 weight가 작아짐
#       (1/count 기반)
train_labels = data_sequences['train']['labels']
class_counts = Counter(train_labels)
total = sum(class_counts.values())

# 역빈도 가중치: 적은 클래스에 더 큰 가중치
# total / (NUM_CLASSES * count) 공식으로 균형
class_weight_map = {
    c: total / (NUM_CLASSES * class_counts.get(c, 1))
    for c in range(NUM_CLASSES)
}
sample_weights = [class_weight_map[l] for l in train_labels]
sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

# [수정] DataLoader 최적화
# num_workers=0: Windows + Jupyter 환경에서 multiprocessing 충돌 방지를 위해
#                안전한 기본값 사용. 안정성 우선이면 0 유지.
#                Linux 환경이면 4로 올리고 persistent_workers=True 권장.
# pin_memory=True: CUDA 사용 시 CPU->GPU 전송 속도 향상
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    collate_fn=collate_sequences,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0)
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
    collate_fn=collate_sequences,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0)
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
    collate_fn=collate_sequences,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0)
)

print(f"\nDataLoader 생성:")
print(f"  Train: {len(train_loader)} batches (bs={BATCH_SIZE})")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")
print(f"  클래스 분포: {dict(class_counts)}")
print(f"  클래스 가중치: { {k: f'{v:.4f}' for k, v in class_weight_map.items()} }")


[1/3] Welford 알고리즘으로 mean/std 계산...


  완료: 40,360,399 프레임, dim=106
  스케일러 저장: seq_scaler_2stage.pkl

[2/3] 청크 파일 정규화 (in-place)...
  checkpoint_seq_train_real_9b06a812_chunks 완료
  checkpoint_seq_train_fake_8a850e9b_chunks 완료
  checkpoint_seq_val_real_f26d8108_chunks 완료
  checkpoint_seq_val_fake_9077ecb0_chunks 완료
  checkpoint_seq_test_real_f26b714d_chunks 완료
  checkpoint_seq_test_fake_f0a7faef_chunks 완료

[3/3] data_sequences 재구성...
  train: 127,111개, 16320 MB
  val: 15,888개, 2040 MB
  test: 15,891개, 2039 MB

정규화 완료

DataLoader 생성:
  Train: 1987 batches (bs=64)
  Val:   125 batches
  Test:  125 batches
  클래스 분포: {0: 38300, 1: 88811}
  클래스 가중치: {0: '1.6594', 1: '0.7156'}


---
## 셀 6: Stage 2 모델 정의 - Transformer / BiLSTM / Hybrid 분류기

3가지 Stage 2 아키텍처를 구현하여 비교:

### A. Transformer Encoder
- Self-attention이 시퀀스 내 임의 위치 간 의존성을 직접 모델링
- 장거리 의존성 포착에 강점 (음성 시작부-중간부 불일치 등)
- 참고: Zaman et al. (2024) "Hybrid transformer architectures"

### B. BiLSTM
- 순방향/역방향 시간 정보를 모두 활용하여 문맥 의존적 표현 학습
- 양방향 코아티큘레이션 패턴 모델링 가능
- 참고: LSTM-AE-DRDE (2025) "attention-enhanced LSTM"

### C. Hybrid (Transformer + BiLSTM)
- Transformer의 글로벌 어텐션("어디서" 아티팩트 발생) +
  LSTM의 순차적 모델링("어떤 순서로" 아티팩트 전개)
- 참고: Xie et al. (2024) EAT, Petmezas et al. (2025) CNN-LSTM-Transformer

### 공통 구성 요소
- **Positional Encoding**: 사인/코사인 (Vaswani et al., 2017)
- **SpecAugment**: 시간/특징 마스킹으로 일반화 향상 (Park et al., 2019)
- **Attention Pooling**: 딥페이크 단서 프레임에 높은 가중치 자동 부여


In [6]:
class PositionalEncoding(nn.Module):
    # 사인/코사인 위치 인코딩 (Vaswani et al., 2017)
    #
    # Transformer는 순서 정보가 없으므로 위치 정보를 명시적으로 주입.
    # 서로 다른 주파수의 사인/코사인 함수로 각 위치에 고유한 벡터 할당.
    #
    # 성질:
    #   PE(pos+k)는 PE(pos)의 선형 함수 -> 상대 위치 학습 용이
    #   학습 불필요한 결정론적 인코딩 (파라미터 추가 없음)
    #   시퀀스 길이에 독립적 (학습보다 긴 시퀀스에도 적용 가능)

    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class SpecAugment(nn.Module):
    # SpecAugment 스타일 데이터 증강 (Park et al., 2019, 학습 시에만 적용)
    #
    # 시간 마스킹: 일부 프레임을 0으로 -> 특정 시간대에 의존하지 않도록
    # 특징 마스킹: 일부 차원을 0으로 -> 특정 특징에 과적합 방지
    # 목적: 모델이 "부분적 정보만으로도 딥페이크를 탐지"하도록 강제

    def __init__(self, time_mask_param=30, freq_mask_param=10,
                 num_time_masks=2, num_freq_masks=2):
        super().__init__()
        self.time_mask_param = time_mask_param
        self.freq_mask_param = freq_mask_param
        self.num_time_masks = num_time_masks
        self.num_freq_masks = num_freq_masks

    def forward(self, x):
        if not self.training:
            return x
        x = x.clone()
        B, T, D = x.shape
        for _ in range(self.num_time_masks):
            t = torch.randint(0, min(self.time_mask_param, T), (1,)).item()
            t0 = torch.randint(0, max(T - t, 1), (1,)).item()
            x[:, t0:t0+t, :] = 0
        for _ in range(self.num_freq_masks):
            f = torch.randint(0, min(self.freq_mask_param, D), (1,)).item()
            f0 = torch.randint(0, max(D - f, 1), (1,)).item()
            x[:, :, f0:f0+f] = 0
        return x


class TransformerClassifier(nn.Module):
    # Transformer Encoder 기반 시퀀스 분류기
    #
    # 아키텍처:
    #   입력 (B,T,D) -> Linear Projection -> Positional Encoding
    #   -> N x Transformer Encoder Layer -> Attention Pooling -> Classification Head
    #
    # Transformer Encoder의 역할:
    #   Self-attention이 모든 프레임 쌍 간의 관계를 직접 모델링.
    #   시간적으로 멀리 떨어진 프레임 간의 불일치도 한 번에 포착 가능.
    #   예: 음성 시작부의 자연스러운 onset과 중간부의 부자연스러운
    #   포먼트 전이 사이의 불일치를 감지.
    #
    # Attention Pooling:
    #   학습 가능한 query 벡터로 시퀀스를 가중 평균.
    #   "딥페이크 단서가 있는 프레임"에 자동으로 높은 가중치 부여.
    #
    # 참고: Zaman et al. (2024) "Hybrid transformer architectures"

    def __init__(self, input_dim, num_classes=NUM_CLASSES, d_model=256,
                 nhead=8, num_layers=4, dim_feedforward=512, dropout=0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model), nn.LayerNorm(d_model),
            nn.ReLU(), nn.Dropout(dropout)
        )
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        self.spec_augment = SpecAugment(30, 20, 2, 2)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward, dropout=dropout,
            activation='gelu',  # GELU: Transformer에서 ReLU보다 우수
            batch_first=True,
            norm_first=True     # Pre-LN: Post-LN보다 학습 안정성 향상
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            norm=nn.LayerNorm(d_model)
        )

        self.attention_pool = nn.Sequential(
            nn.Linear(d_model, d_model // 4), nn.Tanh(),
            nn.Linear(d_model // 4, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.LayerNorm(d_model // 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4), nn.LayerNorm(d_model // 4),
            nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(d_model // 4, num_classes)
        )

    def _create_padding_mask(self, lengths, max_len):
        # 패딩 위치에 True, attention이 부여되지 않도록 마스킹
        mask = torch.arange(max_len, device=lengths.device).unsqueeze(0)
        return mask >= lengths.unsqueeze(1)

    def forward(self, x, lengths=None):
        B, T, D = x.shape
        x = self.spec_augment(x)
        x = self.input_proj(x)
        x = self.pos_encoder(x)

        padding_mask = None
        if lengths is not None:
            padding_mask = self._create_padding_mask(lengths, T)

        x = self.transformer_encoder(x, src_key_padding_mask=padding_mask)

        # Attention Pooling: 딥페이크 단서 프레임에 높은 가중치
        attn_weights = self.attention_pool(x)
        if padding_mask is not None:
            attn_weights = attn_weights.masked_fill(
                padding_mask.unsqueeze(-1), float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=1)
        pooled = torch.sum(x * attn_weights, dim=1)

        return self.classifier(pooled)


class BiLSTMClassifier(nn.Module):
    # 양방향 LSTM 기반 시퀀스 분류기
    #
    # BiLSTM 이론적 근거:
    #   순방향: 과거 프레임의 맥락으로 현재 프레임 해석
    #   역방향: 미래 프레임의 맥락으로 현재 프레임 해석
    #   양방향 결합: 전후 문맥을 모두 고려한 표현
    #
    # 딥페이크 탐지 장점:
    #   TTS의 포먼트 궤적은 "미래 음소에 대한 준비"가 부족할 수 있음.
    #   BiLSTM은 양방향 코아티큘레이션 패턴을 모델링.
    #   Transformer보다 적은 파라미터로 순차적 패턴 포착 가능.
    #
    # 참고: LSTM-AE-DRDE (2025) "attention-enhanced LSTM"

    def __init__(self, input_dim, num_classes=NUM_CLASSES, hidden_dim=256,
                 num_layers=3, dropout=0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout)
        )
        self.spec_augment = SpecAugment(30, 20, 2, 2)

        self.lstm = nn.LSTM(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)

        self.attention_pool = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim // 2), nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2),
            nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x, lengths=None):
        B, T, D = x.shape
        x = self.spec_augment(x)
        x = self.input_proj(x)

        if lengths is not None:
            lengths_cpu = lengths.cpu().clamp(max=T)
            packed = pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=True)
            lstm_out, _ = self.lstm(packed)
            lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True, total_length=T)
        else:
            lstm_out, _ = self.lstm(x)

        lstm_out = self.layer_norm(lstm_out)

        attn_weights = self.attention_pool(lstm_out)
        if lengths is not None:
            mask = torch.arange(T, device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)
            attn_weights = attn_weights.masked_fill(mask.unsqueeze(-1), float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=1)
        pooled = torch.sum(lstm_out * attn_weights, dim=1)

        return self.classifier(pooled)


class HybridTransformerLSTM(nn.Module):
    # Transformer + BiLSTM 하이브리드 분류기
    #
    # 아키텍처:
    #   입력 -> Projection -> PosEnc -> Transformer (글로벌 어텐션)
    #   -> BiLSTM (순차적 모델링) -> Attention Pooling -> Classification
    #
    # 하이브리드 설계 이론적 근거:
    #   1. Transformer: Self-Attention으로 장거리 의존성 포착
    #      -> "어디서" 아티팩트가 있는지 (위치 독립적 관계)
    #   2. BiLSTM: 순차적 패턴 추가 모델링
    #      -> "어떤 순서로" 아티팩트가 나타나는지
    #
    # 참고:
    #   Xie et al. (2024) EAT: ResNet + Transformer + BiLSTM
    #   Petmezas et al. (2025): CNN-LSTM-Transformer hybrid

    def __init__(self, input_dim, num_classes=NUM_CLASSES, d_model=256,
                 nhead=8, num_transformer_layers=2,
                 num_lstm_layers=2, dropout=0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model), nn.LayerNorm(d_model),
            nn.ReLU(), nn.Dropout(dropout)
        )
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        self.spec_augment = SpecAugment(30, 20, 2, 2)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_transformer_layers,
            norm=nn.LayerNorm(d_model)
        )

        # BiLSTM: hidden = d_model//2, bidirectional -> 출력 = d_model
        self.lstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model // 2,
            num_layers=num_lstm_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if num_lstm_layers > 1 else 0
        )
        self.layer_norm = nn.LayerNorm(d_model)

        self.attention_pool = nn.Sequential(
            nn.Linear(d_model, d_model // 4), nn.Tanh(),
            nn.Linear(d_model // 4, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.LayerNorm(d_model // 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x, lengths=None):
        B, T, D = x.shape
        x = self.spec_augment(x)
        x = self.input_proj(x)
        x = self.pos_encoder(x)

        padding_mask = None
        if lengths is not None:
            padding_mask = torch.arange(T, device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)

        x = self.transformer(x, src_key_padding_mask=padding_mask)

        if lengths is not None:
            lengths_cpu = lengths.cpu().clamp(max=T)
            packed = pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=True)
            lstm_out, _ = self.lstm(packed)
            lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True, total_length=T)
        else:
            lstm_out, _ = self.lstm(x)

        lstm_out = self.layer_norm(lstm_out)

        attn_weights = self.attention_pool(lstm_out)
        if padding_mask is not None:
            attn_weights = attn_weights.masked_fill(padding_mask.unsqueeze(-1), float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=1)
        pooled = torch.sum(lstm_out * attn_weights, dim=1)

        return self.classifier(pooled)


# ==============================================================
# 모델 생성
# ==============================================================
FEAT_DIM = data_sequences['train']['sequences'][0].shape[1]
print(f"입력 특징 차원: {FEAT_DIM}")

models = {
    'Transformer': TransformerClassifier(
        input_dim=FEAT_DIM, num_classes=NUM_CLASSES, d_model=256,
        nhead=8, num_layers=4, dim_feedforward=512, dropout=0.3
    ),
    'BiLSTM': BiLSTMClassifier(
        input_dim=FEAT_DIM, num_classes=NUM_CLASSES,
        hidden_dim=256, num_layers=3, dropout=0.3
    ),
    'Hybrid': HybridTransformerLSTM(
        input_dim=FEAT_DIM, num_classes=NUM_CLASSES, d_model=256,
        nhead=8, num_transformer_layers=2, num_lstm_layers=2, dropout=0.3
    )
}

print("\n모델 파라미터 수 비교:")
print("-" * 50)
for name, model in models.items():
    n_params = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  {name:15s}: {n_params:>10,} total, {n_train:>10,} trainable")


입력 특징 차원: 106

모델 파라미터 수 비교:
--------------------------------------------------
  Transformer    :  2,195,011 total,  2,195,011 trainable
  BiLSTM         :  4,466,563 total,  4,466,563 trainable
  Hybrid         :  1,923,587 total,  1,923,587 trainable


---
## 셀 7: 학습 루프 - 다중 모델 학습 및 비교

### 학습 전략
1. **손실 함수**: CrossEntropyLoss + 클래스 가중치 (TTS 3배 강화)
2. **옵티마이저**: AdamW (weight decay가 적응적 학습률과 올바르게 결합)
3. **학습률 스케줄러**: CosineAnnealingWarmRestarts
   - 코사인 곡선으로 감소, 주기적 재설정으로 지역 최솟값 탈출
4. **그래디언트 클리핑**: max_norm=1.0 (RNN/Transformer 폭발 방지)
5. **조기 종료**: patience=20, 기준=Macro F1 (불균형에 강건)


In [10]:
def train_model(model, train_loader, val_loader, model_name,
                num_epochs=NUM_EPOCHS, lr=LEARNING_RATE,
                weight_decay=WEIGHT_DECAY, patience=PATIENCE,
                device=device):
    # Stage 2 모델 학습 함수
    #
    # 학습 전략:
    #   손실: CrossEntropyLoss + 동적 클래스 가중치 (역빈도 기반)
    #   옵티마이저: AdamW (weight decay가 적응적 학습률과 올바르게 결합)
    #   스케줄러: CosineAnnealingWarmRestarts
    #     - 코사인 곡선으로 학습률 감소, 주기적으로 재설정하여 지역 최솟값 탈출
    #   그래디언트 클리핑: max_norm=1.0 (RNN/Transformer 폭발 방지)
    #   조기 종료: patience 에폭 동안 개선 없으면 중단
    #   평가 기준: Macro F1 (클래스 불균형에서 정확도보다 적절한 지표)

    model = model.to(device)

    # [수정] 클래스 가중치 - 학습 데이터의 실제 클래스 분포에서 계산
    # 기존: torch.FloatTensor([1.0, 1.2, 3.0]) - 3-class 하드코딩
    # 수정: NUM_CLASSES에 맞춰 역빈도 가중치 동적 생성
    train_labels_for_weight = data_sequences['train']['labels']
    cls_counts = Counter(train_labels_for_weight)
    total_n = sum(cls_counts.values())
    cw_list = [
        total_n / (NUM_CLASSES * cls_counts.get(c, 1))
        for c in range(NUM_CLASSES)
    ]
    cw = torch.FloatTensor(cw_list).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    history = {
        'train_loss': [], 'train_acc': [], 'val_acc': [],
        'val_f1': [], 'lr': []
    }
    best_val_f1 = 0
    best_val_acc = 0
    patience_counter = 0

    print(f"\n{'='*60}")
    print(f"{model_name} 학습 시작")
    print(f"{'='*60}")

    for epoch in range(num_epochs):
        # --- 학습 ---
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch_seqs, batch_labels, batch_lengths in train_loader:
            batch_seqs = batch_seqs.to(device)
            batch_labels = batch_labels.to(device)
            batch_lengths = batch_lengths.to(device)

            optimizer.zero_grad()
            outputs = model(batch_seqs, batch_lengths)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += batch_labels.size(0)
            train_correct += (predicted == batch_labels).sum().item()

        scheduler.step()
        train_acc = train_correct / train_total
        avg_loss = train_loss / len(train_loader)

        # --- 검증 ---
        model.eval()
        val_preds = []
        val_labels_list = []

        with torch.no_grad():
            for batch_seqs, batch_labels, batch_lengths in val_loader:
                batch_seqs = batch_seqs.to(device)
                batch_lengths = batch_lengths.to(device)
                outputs = model(batch_seqs, batch_lengths)
                _, predicted = torch.max(outputs, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_labels_list.extend(batch_labels.numpy())

        val_acc = accuracy_score(val_labels_list, val_preds)
        val_f1 = f1_score(val_labels_list, val_preds, average='macro')
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(avg_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr'].append(current_lr)

        save_path = f'best_2stage_{model_name.lower()}.pt'
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
            marker = "  [BEST]"
        else:
            patience_counter += 1
            marker = ""

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1:3d}/{num_epochs}] - "
                  f"Loss: {avg_loss:.4f} | Train: {train_acc:.4f} | "
                  f"Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | "
                  f"LR: {current_lr:.6f}{marker}")

        if patience_counter >= patience:
            print(f"\n  조기 종료: {epoch+1} 에폭")
            break

    print(f"\n  학습 완료! 최고 Val Acc: {best_val_acc:.4f}, F1: {best_val_f1:.4f}")
    return history, best_val_acc


# ==============================================================
# 모델 학습 실행
# ==============================================================
MODELS_TO_TRAIN = ['Hybrid', 'Transformer', 'BiLSTM']

all_histories = {}
all_best_accs = {}

for model_name in MODELS_TO_TRAIN:
    model = models[model_name]
    history, best_acc = train_model(
        model, train_loader, val_loader, model_name, device=device
    )
    all_histories[model_name] = history
    all_best_accs[model_name] = best_acc



Hybrid 학습 시작


KeyboardInterrupt: 

In [9]:
import time, torch
from collections import Counter

# 1. 샘플러 동작 확인
print("=== 샘플러 확인 ===")
label_counts = Counter()
for i, (_, labels, _) in enumerate(train_loader):
    label_counts.update(labels.tolist())
    if i >= 50:
        break
print(f"  배치 50개 레이블 분포: {dict(label_counts)}")
print(f"  real:fake 비율 = {label_counts[0]}:{label_counts[1]}")

# 2. train_model 핵심 로직만 단독 실행
print("\n=== train_model 단계별 타이밍 ===")
model = models['Transformer'].to(device)
model.train()

cw = torch.FloatTensor([1.0, 1.0]).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=cw)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()

times = []
for step, (batch_seqs, batch_labels, batch_lengths) in enumerate(train_loader):
    t0 = time.time()

    batch_seqs    = batch_seqs.to(device)
    batch_labels  = batch_labels.to(device)
    batch_lengths = batch_lengths.to(device)
    t1 = time.time()

    optimizer.zero_grad()
    with torch.cuda.amp.autocast():
        out  = model(batch_seqs, batch_lengths)
        loss = criterion(out, batch_labels)
    t2 = time.time()

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    t3 = time.time()

    times.append({
        'transfer': t1 - t0,
        'forward':  t2 - t1,
        'backward': t3 - t2,
        'total':    t3 - t0,
    })

    if step >= 9:
        break

print(f"  {'step':>4}  {'transfer':>9}  {'forward':>8}  {'backward':>9}  {'total':>7}")
for i, t in enumerate(times):
    print(f"  {i:>4}  {t['transfer']*1000:>8.1f}ms  {t['forward']*1000:>7.1f}ms  {t['backward']*1000:>8.1f}ms  {t['total']*1000:>6.1f}ms")

avg_total = sum(t['total'] for t in times) / len(times)
print(f"\n  평균 배치: {avg_total*1000:.1f}ms")
print(f"  3972배치 예상: {avg_total * 3972 / 60:.1f}분/에폭")

=== 샘플러 확인 ===
  배치 50개 레이블 분포: {1: 1629, 0: 1635}
  real:fake 비율 = 1635:1629

=== train_model 단계별 타이밍 ===
  step   transfer   forward   backward    total
     0       0.0ms    139.0ms     109.8ms   248.9ms
     1       1.0ms      6.2ms      32.5ms    39.7ms
     2       1.0ms      6.5ms      31.4ms    38.9ms
     3       0.0ms      5.1ms      34.8ms    39.9ms
     4       1.5ms      8.1ms      30.6ms    40.2ms
     5       2.0ms      6.5ms      30.6ms    39.2ms
     6       0.0ms      7.6ms      31.2ms    38.8ms
     7       1.0ms      6.0ms      31.5ms    38.5ms
     8       1.0ms      6.5ms      31.3ms    38.8ms
     9       1.0ms      6.5ms      31.8ms    39.3ms

  평균 배치: 60.2ms
  3972배치 예상: 4.0분/에폭


---
## 셀 8: 테스트 세트 평가

평가 지표:
- 정확도 / Macro F1 / 클래스별 Precision, Recall, F1 / 혼동 행렬


In [ ]:
def evaluate_model(model, test_loader, model_name, device=device):
    # 테스트 세트 평가: 정확도, Macro F1, 클래스별 P/R/F1, 혼동 행렬

    save_path = f'best_2stage_{model_name.lower()}.pt'
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=device))
    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch_seqs, batch_labels, batch_lengths in test_loader:
            batch_seqs = batch_seqs.to(device)
            batch_lengths = batch_lengths.to(device)
            outputs = model(batch_seqs, batch_lengths)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    report = classification_report(
        all_labels, all_preds, target_names=class_names, digits=4
    )
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n{'='*60}")
    print(f"{model_name} - 테스트 평가")
    print(f"{'='*60}")
    print(f"  정확도: {accuracy:.4f}")
    print(f"  Macro F1: {macro_f1:.4f}")
    print(f"\n{report}")
    print(f"혼동 행렬:\n{cm}")

    return {
        'accuracy': accuracy, 'macro_f1': macro_f1,
        'preds': all_preds, 'labels': all_labels,
        'probs': all_probs, 'cm': cm, 'report': report
    }


all_results = {}
for model_name in MODELS_TO_TRAIN:
    model = models[model_name]
    results = evaluate_model(model, test_loader, model_name, device=device)
    all_results[model_name] = results


NameError: name 'MODELS_TO_TRAIN' is not defined

---
## 셀 9: 결과 시각화

1. 학습 곡선 비교 (Loss, Accuracy, F1, LR)
2. 모델별 혼동 행렬 (정규화)
3. 클래스별 성능 비교 바 차트 (Precision, Recall, F1)


In [ ]:
# ==============================================================
# 1. 학습 곡선 비교
# ==============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = {'Transformer': '#2E86AB', 'BiLSTM': '#A23B72', 'Hybrid': '#F18F01'}

for name, hist in all_histories.items():
    axes[0, 0].plot(hist['train_loss'], label=name,
                     color=colors.get(name, 'gray'), linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

for name, hist in all_histories.items():
    c = colors.get(name, 'gray')
    axes[0, 1].plot(hist['train_acc'], label=f'{name} Train',
                     color=c, linewidth=2, linestyle='-')
    axes[0, 1].plot(hist['val_acc'], label=f'{name} Val',
                     color=c, linewidth=2, linestyle='--')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Accuracy (Train vs Val)', fontweight='bold')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

for name, hist in all_histories.items():
    axes[1, 0].plot(hist['val_f1'], label=name,
                     color=colors.get(name, 'gray'), linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Macro F1')
axes[1, 0].set_title('Validation Macro F1', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

for name, hist in all_histories.items():
    axes[1, 1].plot(hist['lr'], label=name,
                     color=colors.get(name, 'gray'), linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.savefig('training_curves_2stage.png', dpi=150, bbox_inches='tight')
plt.show()

# ==============================================================
# 2. 모델별 혼동 행렬
# ==============================================================

n_models = len(all_results)
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
if n_models == 1:
    axes = [axes]

for idx, (name, result) in enumerate(all_results.items()):
    cm = result['cm']
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[idx], vmin=0, vmax=1)
    axes[idx].set_title(
        f'{name}\nAcc: {result["accuracy"]:.4f}, F1: {result["macro_f1"]:.4f}',
        fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrices_2stage.png', dpi=150, bbox_inches='tight')
plt.show()

# ==============================================================
# 3. 클래스별 성능 비교 바 차트
# ==============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['precision', 'recall', 'f1-score']

for midx, metric in enumerate(metrics):
    model_names_list = list(all_results.keys())
    x = np.arange(len(class_names))
    width = 0.8 / len(model_names_list)

    for i, name in enumerate(model_names_list):
        report_dict = classification_report(
            all_results[name]['labels'], all_results[name]['preds'],
            target_names=class_names, output_dict=True
        )
        values = [report_dict[cn][metric] for cn in class_names]
        axes[midx].bar(x + i * width, values, width, label=name,
                       color=colors.get(name, 'gray'), alpha=0.85)

    axes[midx].set_xlabel('Class')
    axes[midx].set_ylabel(metric.capitalize())
    axes[midx].set_title(f'Class-wise {metric.capitalize()}', fontweight='bold')
    axes[midx].set_xticks(x + width * (len(model_names_list) - 1) / 2)
    axes[midx].set_xticklabels(class_names)
    axes[midx].legend()
    axes[midx].set_ylim(0, 1.05)
    axes[midx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('classwise_performance_2stage.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 셀 10: 최종 성능 요약

In [ ]:
# ==============================================================
# 최종 성능 요약
# ==============================================================

print("=" * 70)
print("2-Stage 딥페이크 음성 탐지 - 최종 성능 요약")
print("=" * 70)

print(f"\n{'모델':<25} {'테스트 정확도':>15} {'Macro F1':>12}")
print("-" * 55)
for name, result in all_results.items():
    print(f"  {name:<23} {result['accuracy']:>15.4f} {result['macro_f1']:>12.4f}")

print("-" * 55)
print(f"\n클래스별 Recall 비교:")
print(f"{'모델':<25}", end="")
for cn in class_names:
    print(f" {cn:>10}", end="")
print()
print("-" * 55)

for name, result in all_results.items():
    report_dict = classification_report(
        result['labels'], result['preds'],
        target_names=class_names, output_dict=True
    )
    print(f"  {name:<23}", end="")
    for cn in class_names:
        print(f" {report_dict[cn]['recall']:>10.4f}", end="")
    print()

best_model_name = max(all_results, key=lambda k: all_results[k]['macro_f1'])
print(f"\n최고 모델: {best_model_name} "
      f"(Macro F1: {all_results[best_model_name]['macro_f1']:.4f})")

print(f"\n저장된 산출물:")
print(f"  seq_scaler_2stage.pkl")
for name in MODELS_TO_TRAIN:
    fname = f'best_2stage_{name.lower()}.pt'
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / (1024 * 1024)
        print(f"  {fname:<35} ({size_mb:.1f} MB)")

if torch.cuda.is_available():
    print(f"\nGPU 메모리: "
          f"할당 {torch.cuda.memory_allocated(0)/1024**3:.2f} GB, "
          f"캐시 {torch.cuda.memory_reserved(0)/1024**3:.2f} GB")

print("\n" + "=" * 70)
print("완료!")
print("=" * 70)


---
## 셀 11: 추론 파이프라인

학습된 2-Stage 모델로 새 오디오 파일 예측:
1. 오디오 로드 -> 다중 음향 특징 (106D/frame)
2. (선택) Stage 1 임베딩 (256D/frame)
3. 특징 결합 + 정규화
4. Stage 2 시퀀스 모델 분류
5. 예측 결과 및 신뢰도 출력


In [ ]:
def load_2stage_pipeline(stage2_model_class, stage2_path,
                         stage1_path='best_asvspoof_model.pt',
                         scaler_path='seq_scaler_2stage.pkl',
                         mfcc_scaler_path='scaler_asvspoof.pkl',
                         input_dim=None, device=None):
    # 2-Stage 추론 파이프라인 로드
    #
    # 흐름: 오디오 -> 다중특징(106D) + Stage1임베딩(256D) -> 정규화
    #       -> Stage2 시퀀스모델 -> 예측
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    pipeline = {'device': device}

    # Stage 1 (선택적)
    if os.path.exists(stage1_path):
        pipeline['stage1'] = Stage1EmbeddingExtractor.from_pretrained(
            stage1_path, input_dim=N_MFCC, device=device
        )
        pipeline['stage1'].eval()
    else:
        pipeline['stage1'] = None

    # MFCC 스케일러
    if os.path.exists(mfcc_scaler_path):
        pipeline['mfcc_scaler'] = joblib.load(mfcc_scaler_path)
    else:
        pipeline['mfcc_scaler'] = None

    # 시퀀스 스케일러
    pipeline['seq_scaler'] = joblib.load(scaler_path)

    # Stage 2 모델
    if input_dim is None:
        input_dim = FEAT_DIM
    stage2 = stage2_model_class(input_dim=input_dim, num_classes=NUM_CLASSES)
    stage2.load_state_dict(torch.load(stage2_path, map_location=device))
    stage2 = stage2.to(device)
    stage2.eval()
    pipeline['stage2'] = stage2

    return pipeline


def predict_audio(audio_path, pipeline):
    # 새로운 오디오 파일에 대해 2-Stage 예측 수행
    #
    # 반환: 예측 클래스, 신뢰도, 클래스별 확률
    device = pipeline['device']

    # 1. 다중 음향 특징
    multi_feat = extract_multi_features_per_frame(audio_path)
    if multi_feat is None:
        return {'error': 'feature extraction failed'}

    # 2. Stage 1 임베딩 (선택적)
    if pipeline['stage1'] is not None:
        y, _ = librosa.load(audio_path, sr=SAMPLE_RATE)
        mfcc_raw = librosa.feature.mfcc(
            y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC,
            n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH
        ).T

        T = min(multi_feat.shape[0], mfcc_raw.shape[0])
        multi_feat = multi_feat[:T]
        mfcc_raw = mfcc_raw[:T]

        if pipeline['mfcc_scaler'] is not None:
            mfcc_raw = pipeline['mfcc_scaler'].transform(mfcc_raw)

        with torch.no_grad():
            mfcc_tensor = torch.FloatTensor(mfcc_raw).unsqueeze(0).to(device)
            embedding = pipeline['stage1'](mfcc_tensor).squeeze(0).cpu().numpy()

        combined = np.concatenate([multi_feat, embedding[:T]], axis=1)
    else:
        combined = multi_feat

    # 3. 정규화
    combined = pipeline['seq_scaler'].transform(combined)

    # 4. Stage 2 예측
    with torch.no_grad():
        seq_tensor = torch.FloatTensor(combined).unsqueeze(0).to(device)
        lengths = torch.LongTensor([combined.shape[0]]).to(device)
        outputs = pipeline['stage2'](seq_tensor, lengths)
        probs = F.softmax(outputs, dim=1).cpu().numpy()[0]
        pred_class = np.argmax(probs)

    return {
        'file': os.path.basename(audio_path),
        'prediction': class_names[pred_class],
        'confidence': float(probs[pred_class]),
        'probabilities': {
            class_names[i]: float(probs[i]) for i in range(len(class_names))
        },
        'num_frames': combined.shape[0],
        'feature_dim': combined.shape[1]
    }


# 추론 테스트
print("2-Stage 추론 파이프라인 테스트")
print("-" * 50)

best_model_class = {
    'Transformer': TransformerClassifier,
    'BiLSTM': BiLSTMClassifier,
    'Hybrid': HybridTransformerLSTM
}[best_model_name]

best_model_path = f'best_2stage_{best_model_name.lower()}.pt'

if os.path.exists(best_model_path):
    pipeline = load_2stage_pipeline(
        best_model_class, best_model_path, input_dim=FEAT_DIM
    )
    print(f"파이프라인 로드 완료 (Stage 2: {best_model_name})")
    print(f"  Stage 1: {'사용' if pipeline['stage1'] is not None else '미사용'}")
    print(f"  입력 차원: {FEAT_DIM}")

    test_dir = os.path.join(BASE_DIR, 'test')
    if os.path.exists(test_dir):
        print("\n샘플 추론:")
        for cls_name in class_names:
            cls_dir = os.path.join(test_dir, cls_name)
            if os.path.exists(cls_dir):
                files = sorted(glob(os.path.join(cls_dir, '*.wav')))[:2]
                for f in files:
                    result = predict_audio(f, pipeline)
                    if 'error' not in result:
                        print(f"  [{cls_name:6s}] {result['file']:30s} -> "
                              f"{result['prediction']:6s} "
                              f"(conf: {result['confidence']:.4f})")
else:
    print(f"모델 파일 '{best_model_path}' 없음")

print("\n모든 작업 완료!")


---
## 셀 12: 아키텍처 요약

```
                    2-Stage 딥페이크 음성 탐지 시스템
                    ================================

 [오디오 입력 (WAV)]
        |
        v
 +----------------------------------------------+
 |     다중 음향 특징 추출 (프레임 단위)           |
 |  MFCC (13D) + Delta (13D) + Delta2 (13D)     |
 |  LFCC (20D) + Mel-Spec (40D) + Contrast (7D) |
 |  합계: 106차원 프레임 벡터                      |
 +----------------------------------------------+
        |                    |
        |                    v
        |     +--------------------------------------+
        |     | Stage 1: SE-Attention 임베딩 추출기   |
        |     | (기존 모델 전이 학습)                  |
        |     | MFCC(13) -> FC+SE -> 256D 임베딩      |
        |     +--------------------------------------+
        |                    |
        v                    v
 +----------------------------------------------+
 |  프레임 특징 결합: 106D + 256D = 362D          |
 +----------------------------------------------+
        |
        v
 +----------------------------------------------+
 |     Stage 2: 시간적 시퀀스 모델링               |
 |  A. Transformer Encoder (글로벌 어텐션)        |
 |  B. BiLSTM (양방향 순차 모델링)                |
 |  C. Hybrid (Transformer + BiLSTM)             |
 +----------------------------------------------+
        |
        v
 +----------------------------------------------+
 |  Attention Pooling -> Classification Head     |
 |  -> 3클래스 (Real / Fake / TTS)               |
 +----------------------------------------------+
```

### 기존 1-Stage와의 핵심 차이점

| 항목 | 1-Stage (기존) | 2-Stage (제안) |
|------|---------------|---------------|
| 입력 | MFCC 13D 평균 벡터 (1D) | 106D 다중특징 시퀀스 (2D) |
| 시간 정보 | 소실 (mean pooling) | 보존 (프레임 시퀀스) |
| 특징 종류 | MFCC만 | MFCC+LFCC+Mel+Contrast+Delta |
| 모델 | FC + SE 어텐션 | Transformer / BiLSTM / Hybrid |
| 전이 학습 | 없음 | Stage 1 임베딩 재활용 |
| 데이터 증강 | SMOTE (벡터 보간) | SpecAugment (시간/특징 마스킹) |
